In [2]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import cv2
import numpy as np
import pickle
import time
import datetime
import gradio as gr
from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity
from ultralytics import YOLO
from insightface.app import FaceAnalysis
import subprocess

from huggingface_hub import hf_hub_download
import shutil, torch, json
from pathlib import Path

# --- configuración global ---
CAMARA_INDEX   = 0
UMBRAL_COSENO  = 0.5
MODELO_YOLO    = "models/yolov12n-face.pt"
WHITELIST_PATH = Path("whitelist/whitelist.pkl")
ALERTS_PATH    = Path("alerts_log")

In [3]:
# busco el modelo localmente primero (si clonaste el repo con Git LFS lo tendrás ahí)
# si no existe, lo descargo automáticamente desde HuggingFace

MODEL_PATH  = Path("models/full_model.pth")
CLASES_PATH = Path("models/clases.json")

if MODEL_PATH.exists():
    print("✅ Modelo encontrado en local (models/full_model.pth)")
else:
    print("⚠️ Modelo no encontrado en local, descargando desde HuggingFace...")
    ruta = hf_hub_download(
        repo_id="mynorhm/security-room-classifier",
        filename="full_model.pth",
        repo_type="model"
    )
    shutil.copy(ruta, MODEL_PATH)
    print("✅ Modelo descargado desde HuggingFace")

if CLASES_PATH.exists():
    print("✅ Clases encontradas en local (models/clases.json)")
else:
    print("⚠️ Clases no encontradas en local, descargando desde HuggingFace...")
    ruta = hf_hub_download(
        repo_id="mynorhm/security-room-classifier",
        filename="clases.json",
        repo_type="model"
    )
    shutil.copy(ruta, CLASES_PATH)
    print("✅ Clases descargadas desde HuggingFace")

# cargo el modelo y las clases
learn_zonas = torch.load(MODEL_PATH, map_location="cpu", weights_only=False)
learn_zonas.eval()

with open(CLASES_PATH) as f:
    clases_zonas = json.load(f)

print(f"Clasificador de zonas listo — clases: {clases_zonas}")

✅ Modelo encontrado en local (models/full_model.pth)
✅ Clases encontradas en local (models/clases.json)
Clasificador de zonas listo — clases: ['bathroom', 'bedroom', 'corridor', 'dining_room', 'garage', 'kitchen', 'livingroom', 'stairscase']


In [4]:
# --- cargo whitelist ---
def cargar_whitelist():
    if WHITELIST_PATH.exists():
        with open(WHITELIST_PATH, "rb") as f:
            return pickle.load(f)
    return {}

def guardar_whitelist(whitelist):
    with open(WHITELIST_PATH, "wb") as f:
        pickle.dump(whitelist, f)

whitelist = cargar_whitelist()
print(f"Todo cargado — Whitelist: {list(whitelist.keys())}")

Todo cargado — Whitelist: ['Mynor']


In [11]:
# mismas funciones de siempre pero limpias para usar desde Gradio
def ver_alertas():
        """muestro las últimas 5 capturas de intrusos guardadas"""
        archivos = sorted(ALERTS_PATH.glob("*.jpg"), reverse=True)[:5]
        if not archivos:
            return [], "No hay alertas registradas"
        imagenes = [cv2.cvtColor(cv2.imread(str(f)), cv2.COLOR_BGR2RGB) for f in archivos]
        nombres  = "\n".join([f.name for f in archivos])
        return imagenes, nombres

def reconocer(frame):
    """detecto y reconozco caras en un frame, devuelvo resultados"""
    faces = insight.get(frame)
    resultados = []

    for face in faces:
        emb          = face.embedding.reshape(1, -1)
        mejor_nombre = "Intruso"
        mejor_score  = 0.0

        for nombre, emb_ref in whitelist.items():
            score = cosine_similarity(emb, emb_ref.reshape(1, -1))[0][0]
            if score > mejor_score:
                mejor_score  = score
                mejor_nombre = nombre if score >= UMBRAL_COSENO else "Intruso"

        resultados.append({
            "nombre"    : mejor_nombre,
            "score"     : mejor_score,
            "bbox"      : face.bbox.astype(int),
            "autorizado": mejor_nombre != "Intruso"
        })

    return resultados

def dibujar(frame, resultados):
    """dibujo bboxes y etiquetas sobre el frame"""
    out = frame.copy()
    for r in resultados:
        x1, y1, x2, y2 = r["bbox"]
        color = (0, 255, 0) if r["autorizado"] else (0, 0, 255)
        label = f"{r['nombre']} ({r['score']:.2f})"

        cv2.rectangle(out, (x1, y1), (x2, y2), color, 2)
        (w, h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
        cv2.rectangle(out, (x1, y1 - h - 10), (x1 + w, y1), color, -1)
        cv2.putText(out, label, (x1, y1 - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
    return out

def registrar_desde_foto(nombre, imagen):
    """registro una persona en la whitelist desde una foto subida en Gradio"""
    if not nombre.strip():
        return "❌ Escribe un nombre primero"

    imagen_bgr = cv2.cvtColor(imagen, cv2.COLOR_RGB2BGR)
    faces      = insight.get(imagen_bgr)

    if not faces:
        return "❌ No detecté ninguna cara en la foto"

    whitelist[nombre] = faces[0].embedding
    guardar_whitelist(whitelist)
    return f"✅ {nombre} registrado en la whitelist"

def eliminar_persona(nombre):
    """elimino una persona de la whitelist"""
    if nombre in whitelist:
        del whitelist[nombre]
        guardar_whitelist(whitelist)
        return f"✅ {nombre} eliminado"
    return f"❌ {nombre} no está en la whitelist"

def personas_en_whitelist():
    """devuelvo la lista actual de personas autorizadas"""
    if not whitelist:
        return "Whitelist vacía"
    return "\n".join([f"✅ {n}" for n in whitelist.keys()])

def ver_logs():
    """leo los últimos 20 logs del proceso del sistema"""
    global proceso_sistema
    if not proceso_sistema:
        return "⚫ Sistema no iniciado"
    
    # leo stdout y stderr del proceso
    try:
        output = ""
        if proceso_sistema.stdout:
            line = proceso_sistema.stdout.readline()
            while line:
                output += line.decode("utf-8")
                line = proceso_sistema.stdout.readline()
        if not output:
            return "Sin logs disponibles"
        return output
    except Exception as e:
        return f"Error leyendo logs: {e}"

print("✅ Funciones listas")

✅ Funciones listas


In [13]:
import subprocess
import signal
import os
import sys

# variable global para controlar el proceso
proceso_sistema = None

def arrancar_sistema():
    global proceso_sistema
    if proceso_sistema and proceso_sistema.poll() is None:
        return "⚠️ El sistema ya está corriendo"
    
    proceso_sistema = subprocess.Popen(
        [sys.executable, "src/sistema.py"],  # uso el mismo python que corre el notebook
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE
    )
    time.sleep(2)
    
    if proceso_sistema.poll() is None:
        return f"✅ Sistema arrancado (PID: {proceso_sistema.pid})"
    else:
        error = proceso_sistema.stderr.read().decode()
        return f"❌ Error arrancando: {error}"
def parar_sistema():
    global proceso_sistema
    if not proceso_sistema or proceso_sistema.poll() is not None:
        return "⚠️ El sistema no está corriendo"
    
    proceso_sistema.terminate()
    proceso_sistema.wait()
    return "🛑 Sistema parado"

def estado_sistema():
    global proceso_sistema
    if not proceso_sistema:
        return "⚫ No iniciado"
    if proceso_sistema.poll() is None:
        return f"🟢 Corriendo (PID: {proceso_sistema.pid})"
    return "🔴 Parado"

with gr.Blocks(title="Sistema de Seguridad") as demo:
    gr.Markdown("# 🔐 Sistema de Seguridad Facial")

    with gr.Tab("🎥 Control del Sistema"):
        gr.Markdown("Arranca y para el sistema de seguridad en vivo")
        estado = gr.Textbox(label="Estado", value="⚫ No iniciado")
        with gr.Row():
            btn_arrancar = gr.Button("▶️ Arrancar sistema", variant="primary")
            btn_parar    = gr.Button("⏹️ Parar sistema", variant="stop")
            btn_estado   = gr.Button("🔄 Actualizar estado")
        btn_arrancar.click(arrancar_sistema, outputs=estado)
        btn_parar.click(parar_sistema, outputs=estado)
        btn_estado.click(estado_sistema, outputs=estado)

    with gr.Tab("👤 Gestionar Whitelist"):
        gr.Markdown("Añade o elimina personas autorizadas")
        with gr.Row():
            with gr.Column():
                foto_nueva    = gr.Image(sources=["webcam", "upload"], label="Foto de la persona")
                nombre_nuevo  = gr.Textbox(label="Nombre")
                btn_registrar = gr.Button("Registrar", variant="primary")
                resultado_reg = gr.Textbox(label="Resultado")
                btn_registrar.click(registrar_desde_foto,
                                    inputs=[nombre_nuevo, foto_nueva],
                                    outputs=resultado_reg)
            with gr.Column():
                nombre_eliminar = gr.Textbox(label="Nombre a eliminar")
                btn_eliminar    = gr.Button("Eliminar", variant="stop")
                resultado_eli   = gr.Textbox(label="Resultado")
                lista_actual    = gr.Textbox(label="Personas autorizadas", lines=5)
                btn_eliminar.click(eliminar_persona,
                                   inputs=nombre_eliminar,
                                   outputs=resultado_eli)
                btn_ver_lista = gr.Button("Actualizar lista")
                btn_ver_lista.click(personas_en_whitelist, outputs=lista_actual)

    with gr.Tab("🚨 Log de Alertas"):
        gr.Markdown("Últimas 5 capturas de intrusos")
        btn_alertas     = gr.Button("Cargar alertas")
        galeria         = gr.Gallery(label="Capturas", columns=3)
        nombres_alertas = gr.Textbox(label="Archivos", lines=5)
        btn_alertas.click(ver_alertas, outputs=[galeria, nombres_alertas])
        
    with gr.Tab("📋 Logs"):
        logs_box    = gr.Textbox(label="Logs del sistema", lines=15)
        btn_logs.click(ver_logs, outputs=logs_box)

demo.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
